# Customer growth scratchpad

Exploring paid-user conversion, spend, and churn by region.

The data is loaded from `data/customers.csv` and everything runs in the
browser — this notebook has no server-side dependencies.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/customers.csv")
df.head()

In [ ]:
# The funnel only counts sessions where checkout was actually shown.
eligible_sessions = df.loc[df["plan"] != "free", "sessions"].sum()
visitors = df["sessions"].sum()

print("visitors:", visitors)
print("eligible_sessions:", eligible_sessions)

## Paid customers only

Everything below looks at customers who are on a paid plan.

In [ ]:
working = df[df["plan"] != "free"].copy()
len(working)

In [ ]:
converted = working["converted"].sum()

# Conversion against the funnel baseline
conversion_rate = converted / visitors

print(f"converted: {converted}")
print(f"conversion rate: {conversion_rate:.1%}")

## Spend and churn by region

In [ ]:
by_region = (
    working.groupby("region")
    .agg(
        customers=("customer_id", "count"),
        total_spend=("monthly_spend", "sum"),
        mean_spend=("monthly_spend", "mean"),
        churn_rate=("churned", "mean"),
    )
    .sort_values("total_spend", ascending=False)
)

by_region

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_region.index, by_region["mean_spend"], color="#4c72b0")
ax.set_title("Mean monthly spend by region (paid plans)")
ax.set_xlabel("Region")
ax.set_ylabel("Mean monthly spend ($)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()